# S02 — SQL Analytics: Window Functions

**Window functions are the single most important SQL skill for data science.** They let you compute rankings, running totals, moving averages, and period comparisons without collapsing rows — which is exactly what analytical queries need.

**Mental model:** A window function looks at a "window" of rows around each row and computes something — without changing the number of output rows. `GROUP BY` collapses; window functions do not.

**Syntax:**
```sql
function() OVER (
    PARTITION BY col1        -- divide into groups
    ORDER BY col2            -- order within group
    ROWS BETWEEN ... AND ... -- define the frame
)
```

**Reference:** [DuckDB Window Functions](https://duckdb.org/docs/sql/window_functions)

**Topics:** ROW_NUMBER, RANK, DENSE_RANK, LAG/LEAD, running totals, moving averages, NTILE, FIRST_VALUE/LAST_VALUE, PERCENT_RANK.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail['CustomerID'] = pd.to_numeric(retail['CustomerID'], errors='coerce')
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['CustomerID'] = retail['CustomerID'].astype(int)

# Monthly aggregate
monthly = retail.groupby('Month').agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index()

con = duckdb.connect()
con.register('retail', retail)
con.register('monthly', monthly)

def q(sql): return con.execute(sql).df()
print("Ready. Tables: retail, monthly")

---
## Exercise 1 — ROW_NUMBER, RANK, DENSE_RANK

**Business question:** For each country, rank customers by total revenue. Show CustomerID, Country, total_revenue, and three columns: `row_num`, `rank`, `dense_rank`. Include only the top-3 customers per country (by row_num). Use `United Kingdom`, `Germany`, `France`, `EIRE`, `Spain` only.

**Key concept to understand:** What is the difference between RANK and DENSE_RANK when two customers have the same revenue? Write your answer in the markdown cell below.

In [ ]:
sql1 = """
-- YOUR SQL HERE
-- Hint: use a CTE with window functions, then filter WHERE row_num <= 3
"""

result1 = q(sql1)
result1

In [ ]:
# --- ASSERTIONS ---
countries = ['United Kingdom','Germany','France','EIRE','Spain']
assert set(result1['Country'].unique()).issubset(set(countries))
# Max 3 per country
assert result1.groupby('Country').size().max() <= 3
for col in ['row_num','rank','dense_rank']:
    assert col in result1.columns.str.lower().tolist() or col in result1.columns.tolist(), f"Missing: {col}"
print("✓ Exercise 1 passed")
print(result1.to_string(index=False))

**RANK vs DENSE_RANK:** *(Write the difference here. When would you use each?)*

---
## Exercise 2 — LAG & LEAD: Period Comparisons

**Business question:** For each month in the `monthly` table, compute:
- `prev_revenue`: previous month's revenue (LAG)
- `next_revenue`: next month's revenue (LEAD)
- `mom_change`: current - previous (NULL for first month)
- `mom_pct`: (current - previous) / previous * 100, rounded to 2dp
- `yoy_change`: revenue vs same month 12 months prior (LAG with offset=12)

**Key concept:** What does `LAG(col, 2)` return? What about `LAG(col, 1, 0)` — what is the third argument?

In [ ]:
sql2 = """
-- YOUR SQL HERE
"""

result2 = q(sql2)
result2

In [ ]:
# --- ASSERTIONS ---
assert len(result2) == len(monthly)
assert pd.isna(result2['prev_revenue'].iloc[0]), "First prev_revenue must be NULL"
assert pd.isna(result2['next_revenue'].iloc[-1]), "Last next_revenue must be NULL"
# MoM calc check on row 2
r1 = float(monthly['Revenue'].iloc[0])
r2 = float(monthly['Revenue'].iloc[1])
expected_mom = round((r2 - r1) / r1 * 100, 2)
assert abs(float(result2['mom_pct'].iloc[1]) - expected_mom) < 0.05
print("✓ Exercise 2 passed")
print(result2.to_string(index=False))

**LAG third argument:** *(What does `LAG(col, 1, 0)` do differently from `LAG(col, 1)`?)*

---
## Exercise 3 — Running Totals & Cumulative Aggregates

**Business question:** For each month, compute:
- `cumulative_revenue`: running total of revenue from the first month
- `cumulative_pct`: what % of all-time revenue has been earned by that month
- `running_avg_revenue`: rolling average over all months up to and including current
- `running_max`: the highest revenue achieved in any month up to and including current

**Frame concept:** `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — know what each part means.

In [ ]:
sql3 = """
-- YOUR SQL HERE
"""

result3 = q(sql3)
result3

In [ ]:
# --- ASSERTIONS ---
total_rev = monthly['Revenue'].sum()
assert abs(float(result3['cumulative_revenue'].iloc[-1]) - total_rev) < 1
assert abs(float(result3['cumulative_pct'].iloc[-1]) - 100) < 0.01
assert result3['cumulative_revenue'].is_monotonic_increasing
assert result3['running_max'].is_monotonic_increasing
print("✓ Exercise 3 passed")
print(result3.to_string(index=False))

---
## Exercise 4 — Moving Averages: Frame Control

**Business question:** Compute three moving averages on monthly revenue:
- `ma_3`: 3-month simple moving average (`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`)
- `ma_6`: 6-month moving average
- `ema_smooth`: exponentially weighted — approximate it as a weighted average where current month weight=3, previous=2, two months ago=1 (normalize weights). Use only 3-row window.

Also compute `volatility_3m`: standard deviation of revenue over the same 3-month window.

**Key concept:** What is the difference between `ROWS` and `RANGE` frame mode? Write your answer below.

In [ ]:
sql4 = """
-- YOUR SQL HERE
"""

result4 = q(sql4)
result4

In [ ]:
# --- ASSERTIONS ---
assert len(result4) == len(monthly)
# First row MA should equal first revenue (only 1 row in window)
r0 = float(monthly['Revenue'].iloc[0])
assert abs(float(result4['ma_3'].iloc[0]) - r0) < 1
# Third row MA_3 should be average of first 3
avg3 = float(monthly['Revenue'].iloc[:3].mean())
assert abs(float(result4['ma_3'].iloc[2]) - avg3) < 1
print("✓ Exercise 4 passed")
print(result4[['Month','Revenue','ma_3','ma_6','volatility_3m']].to_string(index=False))

**ROWS vs RANGE:** *(Explain the difference with an example of when they produce different results.)*

---
## Exercise 5 — NTILE & Percentile Ranking

**Business question:** Segment customers into revenue quartiles (NTILE 4) and deciles (NTILE 10). For each quartile, show: quartile number, n_customers, min_revenue, max_revenue, total_revenue, pct_of_total_revenue, and `cumulative_revenue_pct` (running total of pct_of_total_revenue).

This is the **Lorenz curve in SQL** — a key inequality measure.

In [ ]:
sql5 = """
-- YOUR SQL HERE
-- Step 1: CTE to compute total revenue per customer
-- Step 2: CTE to assign NTILE(4) and NTILE(10)
-- Step 3: Aggregate by quartile with cumulative pct
"""

result5 = q(sql5)
result5

In [ ]:
# --- ASSERTIONS ---
assert len(result5) == 4, "Must have 4 quartile rows"
quartile_col = result5.columns[0]
assert list(result5[quartile_col]) == [1,2,3,4]
pct_col = [c for c in result5.columns if 'pct' in c.lower() and 'cum' not in c.lower()][0]
assert abs(result5[pct_col].sum() - 100) < 0.1
cum_col = [c for c in result5.columns if 'cum' in c.lower()][0]
assert result5[cum_col].is_monotonic_increasing
assert abs(float(result5[cum_col].iloc[-1]) - 100) < 0.1
print("✓ Exercise 5 passed")
print(result5.to_string(index=False))

---
## Exercise 6 — FIRST_VALUE & LAST_VALUE

**Business question:** For each customer, show every order alongside:
- `first_order_revenue`: revenue of their very first order
- `last_order_revenue`: revenue of their most recent order
- `best_order_revenue`: their highest single-order revenue ever
- `revenue_vs_first`: current order revenue / first order revenue (growth multiple)

Filter to customers with at least 3 orders. Return top 100 rows by CustomerID.

**Key concept:** Why does LAST_VALUE require `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`? Write the answer below.

In [ ]:
# Build invoice-level revenue first
con.execute("""
CREATE OR REPLACE TABLE invoice_revenue AS
SELECT
    InvoiceNo,
    CustomerID,
    MIN(InvoiceDate) as invoice_date,
    SUM(Revenue) as invoice_revenue
FROM retail
GROUP BY InvoiceNo, CustomerID
""")

sql6 = """
-- YOUR SQL HERE
-- Use invoice_revenue table
"""

result6 = q(sql6)
result6.head()

In [ ]:
# --- ASSERTIONS ---
assert len(result6) <= 100
for col in ['first_order_revenue','last_order_revenue','best_order_revenue','revenue_vs_first']:
    assert col in result6.columns
assert (result6['best_order_revenue'] >= result6['first_order_revenue']).all()
assert (result6['best_order_revenue'] >= result6['last_order_revenue']).all()
print("✓ Exercise 6 passed")
print(result6[['CustomerID','invoice_revenue','first_order_revenue','last_order_revenue','revenue_vs_first']].head())

**LAST_VALUE frame:** *(Why does LAST_VALUE need the extended frame? What does it return with the default frame?)*

---
## Exercise 7 — Gaps & Islands Problem

**Business question:** Find "active streaks" for each customer — consecutive months where they made at least one purchase. For each streak, return: CustomerID, streak_start, streak_end, streak_length_months.

**This is a classic SQL interview problem.** The gaps-and-islands pattern uses ROW_NUMBER subtraction to identify consecutive sequences.

**Hint:** 
1. Create a table of (CustomerID, Month) where customer was active
2. ROW_NUMBER() over customer ordered by month gives a sequential counter
3. ROW_NUMBER() over all months gives another counter
4. Their difference is constant within a consecutive streak

In [ ]:
sql7 = """
-- YOUR SQL HERE
-- This is challenging — think through the gaps-and-islands approach step by step
"""

result7 = q(sql7)
result7.head(10)

In [ ]:
# --- ASSERTIONS ---
assert len(result7) > 0
for col in ['CustomerID','streak_start','streak_end','streak_length_months']:
    assert col in result7.columns, f"Missing: {col}"
assert (result7['streak_length_months'] >= 1).all()
# A streak of length 1 means start == end
single = result7[result7['streak_length_months'] == 1]
if len(single) > 0:
    assert (single['streak_start'] == single['streak_end']).all()
print(f"✓ Exercise 7 passed — {len(result7)} streaks found")
print(f"Longest streak: {result7['streak_length_months'].max()} months")
print(result7.nlargest(5, 'streak_length_months').to_string(index=False))

---
## Exercise 8 — Cohort Retention in Pure SQL

**Business question:** Build a cohort retention matrix entirely in SQL. For each cohort (acquisition month), compute retention rate at periods 0, 1, 2, 3 months.

- Period 0 = acquisition month (always 100%)
- Period N = % of cohort still active N months after acquisition

Return a matrix where rows = cohort months, columns = period_0 through period_3.

In [ ]:
sql8 = """
-- YOUR SQL HERE
-- Hint: PIVOT or conditional aggregation (SUM(CASE WHEN period = N THEN 1 END))
"""

result8 = q(sql8)
result8

In [ ]:
# --- ASSERTIONS ---
assert len(result8) > 0
# Period 0 should always be 100%
period0_col = [c for c in result8.columns if '0' in str(c)][0]
assert result8[period0_col].dropna().eq(100).all() or result8[period0_col].dropna().eq(1.0).all()
# Retention must be between 0 and 100 (or 0 and 1)
numeric_cols = result8.select_dtypes(include='number').columns
for col in numeric_cols:
    vals = result8[col].dropna()
    assert (vals >= 0).all() and (vals <= 100).all(), f"{col} values out of range"
print(f"✓ Exercise 8 passed — {len(result8)} cohorts")
print(result8.to_string(index=False))

---
## Exercise 9 — QUALIFY: DuckDB-specific Filter on Window

**Business question:** Find the single best-selling product (by quantity) for each country, without using a subquery or CTE. Use DuckDB's `QUALIFY` clause.

**QUALIFY** filters rows based on window function results — like HAVING but for window functions.

Return: Country, StockCode, Description, total_quantity, revenue, country_rank.

In [ ]:
sql9 = """
-- YOUR SQL HERE
-- Use QUALIFY RANK() OVER (...) = 1
"""

result9 = q(sql9)
result9.head(10)

In [ ]:
# --- ASSERTIONS ---
# One row per country
assert result9['Country'].nunique() == len(result9), "Must have exactly one product per country"
rank_col = [c for c in result9.columns if 'rank' in c.lower()]
if rank_col:
    assert (result9[rank_col[0]] == 1).all()
print(f"✓ Exercise 9 passed — {len(result9)} countries with top products")
print(result9[['Country','StockCode','Description']].head(8).to_string(index=False))

---
## Exercise 10 — Capstone: Full Analytical Report in SQL

**Business question:** Build a complete monthly business report in a single SQL query using 5+ CTEs and at least 6 different window functions.

The report must include, for each month:
- Revenue, orders, customers (base metrics)
- MoM and YoY revenue change %
- 3-month moving average revenue
- Revenue rank among all months (1 = best)
- Cumulative revenue and % of total
- `performance_band`: `'Top 25%'`, `'Mid 50%'`, `'Bottom 25%'` using NTILE(4) → map 1=Bottom, 2-3=Mid, 4=Top
- `is_record_month`: TRUE if that month is the highest revenue seen so far (running max)

Sort by Month ascending.

In [ ]:
sql10 = """
-- YOUR SQL HERE — at least 5 CTEs, 6 window functions
"""

result10 = q(sql10)
result10

In [ ]:
# --- ASSERTIONS ---
assert len(result10) == len(monthly)
cols_lower = result10.columns.str.lower().tolist()
for required in ['revenue','performance_band','is_record_month']:
    assert any(required in c for c in cols_lower), f"Missing: {required}"

band_col = [c for c in result10.columns if 'band' in c.lower() or 'performance' in c.lower()][0]
assert set(result10[band_col]).issubset({'Top 25%','Mid 50%','Bottom 25%'})

record_col = [c for c in result10.columns if 'record' in c.lower()][0]
# First month should be a record (nothing before it)
assert result10[record_col].iloc[0] in (True, 1, 'true', 'True')

cum_col = [c for c in result10.columns if 'cumul' in c.lower()]
if cum_col:
    assert result10[cum_col[0]].is_monotonic_increasing

print(f"✓ Exercise 10 passed — Full report: {len(result10)} months")
print(result10[[band_col, record_col]].value_counts())